# Suicide & Depression Post Detection

**Dataset**: [Suicide Watch](https://www.kaggle.com/datasets/nikhileswarkomati/suicide-watch)
— 232,074 Reddit posts labelled `suicide` or `non-suicide`


In [7]:
# ── ── 0. Setup & Installation ────────────────────────────────────────────────
# Cell 0a — Install dependencies
!pip install -q groq
# !pip install -q -r requirements.txt


import os, sys, warnings
warnings.filterwarnings('ignore')
# Add the project root to the system path to allow importing from 'src'
sys.path.insert(0, '.') # Reverting to adding the current directory

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Download NLTK resources required by src modules *before* importing them
import nltk
nltk.download('stopwords')

# Project modules
from src.preprocessing import download_nltk_resources, preprocess_dataframe, preprocess_text
from src.features import (
    build_tfidf_vectorizer, build_feature_matrix,
    extract_handcrafted_features, HANDCRAFTED_FEATURE_NAMES,
    save_vectorizer,
)
from src.train_ml import split_data, train_all_models, save_models
from src.evaluate import full_evaluation, compare_models

# Ensure src.llm is reloaded to get the latest function definition
import importlib
if 'src.llm' in sys.modules:
    del sys.modules['src.llm']
import src.llm
importlib.reload(src.llm)
from src.llm import explain_prediction

os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

download_nltk_resources()
print("✓ Setup complete")

✓ Setup complete


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nikhileswarkomati/suicide-watch -p data/ --unzip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nikhileswarkomati/suicide-watch
License(s): CC-BY-SA-4.0
100% 60.6M/60.6M [00:00<00:00, 104MB/s]



## 1. Data Loading
Download the Suicide Watch dataset from Kaggle using the Kaggle API.

In [8]:
# ── ── 1. Data Loading ────────────────────────────────────────────────────────
# ── Option B: Manual upload ───────────────────────────────────────────────────
# Upload the CSV manually and place it at: data/Suicide_Detection.csv

DATA_FILE = 'data/Suicide_Detection.csv'


df_raw = pd.read_csv(DATA_FILE)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(df_raw.head(3))

Dataset loaded: 232,074 rows × 3 columns
   Unnamed: 0                                               text        class
0           2  Ex Wife Threatening SuicideRecently I left my ...      suicide
1           3  Am I weird I don't get affected by compliments...  non-suicide
2           4  Finally 2020 is almost over... So I can never ...  non-suicide


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── ── 2a. Basic Dataset Info ──────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"\nShape          : {df_raw.shape}")
print(f"Columns        : {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nDuplicate rows : {df_raw.duplicated().sum():,}")
print(f"\nBasic statistics (text length):")
df_raw['text_len'] = df_raw['text'].astype(str).apply(len)
print(df_raw['text_len'].describe())

DATASET OVERVIEW

Shape          : (232074, 3)
Columns        : ['Unnamed: 0', 'text', 'class']

Data types:
Unnamed: 0     int64
text          object
class         object
dtype: object

Missing values:
Unnamed: 0    0
text          0
class         0
dtype: int64

Duplicate rows : 0

Basic statistics (text length):
count    232074.000000
mean        689.639736
std        1156.334007
min           3.000000
25%         138.000000
50%         315.000000
75%         801.000000
max       40297.000000
Name: text_len, dtype: float64


In [ ]:
# ── ── 2b. Target / Label Distribution ───────────────────────────────────────
label_counts = df_raw['class'].value_counts()
print("\nClass Distribution:")
print(label_counts)
print(f"Imbalance ratio: {label_counts.iloc[0]/label_counts.iloc[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color=['#4C6EF5', '#F03E3E'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=['#4C6EF5', '#F03E3E'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportions', fontsize=14)

fig.tight_layout()
fig.savefig('outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Class distribution plot saved")


Class Distribution:
class
suicide        116037
non-suicide    116037
Name: count, dtype: int64
Imbalance ratio: 1.00:1
✓ Class distribution plot saved


In [ ]:
# ── ── 2c. Text Length Analysis ───────────────────────────────────────────────
df_raw['word_count'] = df_raw['text'].astype(str).apply(lambda x: len(x.split()))
df_raw['char_count'] = df_raw['text'].astype(str).apply(len)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (col, title) in enumerate([
    ('word_count', 'Word Count Distribution'),
    ('char_count', 'Character Count Distribution')
]):
    for j, (cls, color) in enumerate([('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
        data = df_raw[df_raw['class'] == cls][col]
        axes[i][j].hist(data.clip(upper=data.quantile(0.99)), bins=50,
                         color=color, alpha=0.75, edgecolor='white')
        axes[i][j].set_title(f'{title} — {cls}', fontsize=12)
        axes[i][j].set_xlabel(col.replace('_', ' ').title())
        axes[i][j].set_ylabel('Frequency')
        axes[i][j].axvline(data.median(), color='black', linestyle='--',
                            label=f'Median: {data.median():.0f}')
        axes[i][j].legend()

fig.suptitle('Text Length Analysis by Class', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig('outputs/plots/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Text length analysis saved")

✓ Text length analysis saved


In [ ]:
# ── ── 2d. Boxplot — Word Count by Class ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
df_raw.boxplot(column='word_count', by='class', ax=ax,
               boxprops=dict(color='#4C6EF5'),
               medianprops=dict(color='#F03E3E', linewidth=2),
               whiskerprops=dict(color='#4C6EF5'),
               capprops=dict(color='#4C6EF5'),
               flierprops=dict(marker='o', color='gray', alpha=0.3, markersize=3))
ax.set_title('Word Count by Class (Boxplot)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Word Count')
plt.suptitle('')
fig.tight_layout()
fig.savefig('outputs/plots/boxplot_wordcount.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Boxplot saved")

✓ Boxplot saved


In [ ]:
# ── ── 2e. Word Clouds ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (cls, color) in zip(axes, [('suicide', 'Reds'), ('non-suicide', 'Blues')]):
    texts = ' '.join(df_raw[df_raw['class'] == cls]['text'].astype(str).tolist())
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap=color,
        max_words=150,
        collocations=False,
    ).generate(texts)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud — {cls}', fontsize=14, pad=10)

fig.suptitle('Most Frequent Words by Class', fontsize=15)
fig.tight_layout()
fig.savefig('outputs/plots/wordclouds.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Word clouds saved")

✓ Word clouds saved


In [ ]:
# ── ── 2f. Top N Words ────────────────────────────────────────────────────────
import re
from nltk.corpus import stopwords

STOP = set(stopwords.words('english'))

def top_words(series, n=20):
    words = []
    for text in series.astype(str):
        tokens = re.findall(r'\b[a-z]{3,}\b', text.lower())
        words.extend([t for t in tokens if t not in STOP])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (cls, color) in zip(axes, [('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
    words, counts = zip(*top_words(df_raw[df_raw['class'] == cls]['text']))
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.8)
    ax.set_title(f'Top 20 Words — {cls}', fontsize=13)
    ax.set_xlabel('Frequency')

fig.tight_layout()
fig.savefig('outputs/plots/top_words.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Top words plot saved")

✓ Top words plot saved


In [ ]:
# ── ── 2g. N-gram Analysis ────────────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(series, n=2, top=15):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english', max_features=5000)
    X = vec.fit_transform(series.astype(str))
    counts = X.sum(axis=0).A1
    vocab  = vec.get_feature_names_out()
    return sorted(zip(vocab, counts), key=lambda x: -x[1])[:top]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (cls, color) in enumerate([('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
    data = df_raw[df_raw['class'] == cls]['text']
    for col, n in enumerate([2, 3]):
        ngrams_top = top_ngrams(data, n=n, top=15)
        phrases, counts = zip(*ngrams_top)
        axes[row][col].barh(phrases[::-1], counts[::-1], color=color, alpha=0.8)
        axes[row][col].set_title(
            f'Top {"Bigrams" if n==2 else "Trigrams"} — {cls}', fontsize=12
        )

fig.suptitle('N-gram Analysis by Class', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig('outputs/plots/ngrams.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ N-gram plots saved")

✓ N-gram plots saved


## 3. Data Quality Checks

In [ ]:
# ── ── 3. Data Quality ────────────────────────────────────────────────────────
print("=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

# Skewness
for col in ['word_count', 'char_count']:
    skew = df_raw[col].skew()
    print(f"\nSkewness of {col}: {skew:.3f} "
          f"({'right-skewed' if skew > 0 else 'left-skewed'})")

# Outlier detection using IQR
print("\nOutlier Detection (IQR method):")
for col in ['word_count', 'char_count']:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_raw[(df_raw[col] < Q1 - 1.5*IQR) | (df_raw[col] > Q3 + 1.5*IQR)]
    print(f"  {col}: {len(outliers):,} outliers ({len(outliers)/len(df_raw)*100:.1f}%)")

# Very short / empty posts
short = df_raw[df_raw['word_count'] < 3]
print(f"\nPosts with < 3 words: {len(short):,}")

DATA QUALITY CHECKS

Skewness of word_count: 8.097 (right-skewed)

Skewness of char_count: 8.520 (right-skewed)

Outlier Detection (IQR method):
  word_count: 20,043 outliers (8.6%)
  char_count: 20,362 outliers (8.8%)

Posts with < 3 words: 186


## 4. Preprocessing

In [9]:
# ── ── 4. Preprocessing ───────────────────────────────────────────────────────
print("Applying preprocessing pipeline …")
df = preprocess_dataframe(df_raw, text_col='text', label_col='class')
print(f"\nShape after preprocessing: {df.shape}")
print(df.head(3))

# Save cleaned dataset for Gradio app
df.to_csv('data/suicide_detection.csv', index=False)
print("✓ Cleaned dataset saved → data/suicide_detection.csv")

Applying preprocessing pipeline …
Applying preprocessing pipeline … (this may take a moment)
Preprocessing complete. Rows retained: 231,983

Shape after preprocessing: (231983, 3)
                                                text  label  \
0  Ex Wife Threatening SuicideRecently I left my ...      1   
1  Am I weird I don't get affected by compliments...      0   
2  Finally 2020 is almost over... So I can never ...      0   

                                          clean_text  
0  ex wife threatening suiciderecently left wife ...  
1  weird dont get affected compliment coming some...  
2  finally almost never hear bad year ever swear ...  
✓ Cleaned dataset saved → data/suicide_detection.csv


## 5. Feature Engineering

In [10]:
# ── ── 5. Feature Engineering ─────────────────────────────────────────────────
tfidf = build_tfidf_vectorizer(max_features=50_000, ngram_range=(1, 2))

X = build_feature_matrix(
    clean_texts=df['clean_text'],
    raw_texts=df['text'],
    tfidf=tfidf,
    fit=True,
)
y = df['label'].values

print(f"Feature matrix shape : {X.shape}")
print(f"Label distribution   : {dict(zip(*np.unique(y, return_counts=True)))}")

save_vectorizer(tfidf, 'models/tfidf_vectorizer.joblib')

# Handcrafted feature distributions
hand_feat = extract_handcrafted_features(df['text'])
hand_df = pd.DataFrame(hand_feat, columns=HANDCRAFTED_FEATURE_NAMES)
hand_df['label'] = y

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(HANDCRAFTED_FEATURE_NAMES):
    for cls, color in [(0, '#4C6EF5'), (1, '#F03E3E')]:
        axes[i].hist(hand_df[hand_df['label']==cls][col],
                     bins=30, alpha=0.6, color=color,
                     label='Non-Suicide' if cls==0 else 'Suicide')
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

fig.suptitle('Handcrafted Feature Distributions by Class', fontsize=15)
fig.tight_layout()
fig.savefig('outputs/plots/handcrafted_features.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Handcrafted feature distributions saved")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = hand_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Handcrafted Feature Correlation Matrix', fontsize=13)
fig.tight_layout()
fig.savefig('outputs/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Correlation heatmap saved")

Feature matrix shape : (231983, 50012)
Label distribution   : {np.int64(0): np.int64(115960), np.int64(1): np.int64(116023)}
Vectorizer saved → models/tfidf_vectorizer.joblib
✓ Handcrafted feature distributions saved
✓ Correlation heatmap saved


## 6. Train / Test Split

In [ ]:
# ── ── 6. Train / Test Split ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.20)

print(f"Training set  : {X_train.shape[0]:,} samples")
print(f"Test set      : {X_test.shape[0]:,} samples")
print(f"Train positives: {y_train.sum():,}  ({y_train.mean()*100:.1f}%)")
print(f"Test  positives: {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")

Training set  : 185,586 samples
Test set      : 46,397 samples
Train positives: 92,818  (50.0%)
Test  positives: 23,205  (50.0%)


## 7. Model Training

In [ ]:
# ── ── 7. Model Training ───────────────────────────────────────────────────────
# tune=True to run RandomizedSearchCV
trained_models = train_all_models(X_train, y_train, tune=False)
save_models(trained_models, directory='models/')
print("\n✓ All models trained and saved")


───────────────────────────────────────────────────────
  Training: Logistic Regression
  CV F1  : 0.9144 ± 0.0017
  ✓ Logistic Regression trained

───────────────────────────────────────────────────────
  Training: SVM (LinearSVC)
  CV F1  : 0.8705 ± 0.0086
  ✓ SVM (LinearSVC) trained

───────────────────────────────────────────────────────
  Training: Decision Tree
  CV F1  : 0.8714 ± 0.0010
  ✓ Decision Tree trained

───────────────────────────────────────────────────────
  Training: Random Forest
  CV F1  : 0.8213 ± 0.0025
  ✓ Random Forest trained

───────────────────────────────────────────────────────
  Training: AdaBoost
  CV F1  : 0.8572 ± 0.0014
  ✓ AdaBoost trained
Saved: models/Logistic_Regression.joblib
Saved: models/SVM_LinearSVC.joblib
Saved: models/Decision_Tree.joblib
Saved: models/Random_Forest.joblib
Saved: models/AdaBoost.joblib

✓ All models trained and saved


## 8. BERT Model Loading

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load pre-trained BERT tokenizer and model
print("Loading BERT tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("Loading BERT model...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Move model to GPU if available
if torch.cuda.is_available():
    device = torch.device("cuda")
    model.cuda()
    print("BERT model moved to GPU.")
else:
    device = torch.device("cpu")
    print("BERT model is running on CPU.")

print("✓ BERT tokenizer and model loaded successfully.")

Loading BERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model is running on CPU.
✓ BERT tokenizer and model loaded successfully.


In [4]:
import torch.nn.functional as F
import numpy as np

def predict_with_bert(text):
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    probabilities = F.softmax(logits, dim=1).cpu().numpy()[0]
    predicted_class_id = np.argmax(probabilities)

    confidence = probabilities[predicted_class_id] * 100
    label_str = "Suicide Risk" if predicted_class_id == 1 else "Non-Suicide"

    return label_str, confidence

# Define sample_texts here as it was missing from this cell's scope
sample_texts = [
    "I've been feeling so hopeless lately. I don't see any reason to keep going on.",
    "Just had a great workout! Feeling energized and ready to tackle the day.",
]

print("\nTesting BERT predictions on sample texts:")
for text in sample_texts:
    label_str, conf = predict_with_bert(text)
    print(f"\n{'─'*60}")
    print(f"Text      : {text[:80]}…")
    print(f"Prediction: {label_str}  ({conf:.1f}%)")


Testing BERT predictions on sample texts:

────────────────────────────────────────────────────────────
Text      : I've been feeling so hopeless lately. I don't see any reason to keep going on.…
Prediction: Suicide Risk  (55.8%)

────────────────────────────────────────────────────────────
Text      : Just had a great workout! Feeling energized and ready to tackle the day.…
Prediction: Suicide Risk  (57.2%)


## 9. Evaluation

In [ ]:
# ── ── 8. Evaluation ──────────────────────────────────────────────────────────
comparison_df = full_evaluation(trained_models, X_test, y_test)

print("\n" + "="*55)
print("FINAL COMPARISON TABLE (sorted by F1-Score)")
print("="*55)
print(comparison_df.to_string(float_format="{:.4f}".format))

best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]
print(f"\n Best model: {best_model_name}")


════════════════════════════════════════════════════════════
  MODEL EVALUATION REPORT
════════════════════════════════════════════════════════════

▸ Logistic Regression
  Accuracy  : 0.9144
  Precision : 0.9279
  Recall    : 0.8987
  F1-Score  : 0.9131
  ROC-AUC   : 0.9692

              precision    recall  f1-score   support

 Non-Suicide       0.90      0.93      0.92     23192
     Suicide       0.93      0.90      0.91     23205

    accuracy                           0.91     46397
   macro avg       0.91      0.91      0.91     46397
weighted avg       0.91      0.91      0.91     46397


▸ SVM (LinearSVC)
  Accuracy  : 0.8730
  Precision : 0.9065
  Recall    : 0.8318
  F1-Score  : 0.8676
  ROC-AUC   : 0.9453

              precision    recall  f1-score   support

 Non-Suicide       0.84      0.91      0.88     23192
     Suicide       0.91      0.83      0.87     23205

    accuracy                           0.87     46397
   macro avg       0.88      0.87      0.87     4639

## 10. LLM Explanation (Groq)

In [2]:
import joblib

GROQ_API_KEY = "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"
LLM_MODEL_TO_USE="openai/gpt-oss-20b"

# Load tfidf vectorizer and best model
tfidf = joblib.load('models/tfidf_vectorizer.joblib')
best_model = joblib.load('models/Logistic_Regression.joblib')

sample_texts = [
    "I've been feeling so hopeless lately. I don't see any reason to keep going on.",
    "Just had a great workout! Feeling energized and ready to tackle the day.",
]

for text in sample_texts:
    clean = preprocess_text(text)

    # Build feature for single sample
    from scipy.sparse import csr_matrix
    clean_s = pd.Series([clean])
    raw_s   = pd.Series([text])
    X_sample = build_feature_matrix(clean_s, raw_s, tfidf, fit=False)

    pred = best_model.predict(X_sample)[0]
    if hasattr(best_model, 'predict_proba'):
        conf = best_model.predict_proba(X_sample)[0][pred] * 100
    else:
        conf = 85.0  # fallback

    label_str = "Suicide Risk" if pred == 1 else "Non-Suicide"

    print(f"\n{'─'*60}")
    print(f"Text      : {text[:80]}…")
    print(f"Prediction: {label_str}  ({conf:.1f}%)")

    if GROQ_API_KEY:
        print("\nGroq Explanation:")
        explanation = explain_prediction(text, label_str, conf, api_key=GROQ_API_KEY,model_name=LLM_MODEL_TO_USE)
        print(explanation)
    else:
        print("\n⚠️  Set GROQ_API_KEY to see the LLM explanation.")


────────────────────────────────────────────────────────────
Text      : I've been feeling so hopeless lately. I don't see any reason to keep going on.…
Prediction: Suicide Risk  (94.4%)

Groq Explanation:
The model flagged the post as **Suicide Risk** because it contains several high‑risk linguistic cues. The phrase *“feeling so hopeless”* signals a pervasive sense of despair, while *“I don't see any reason to keep going on”* directly expresses a loss of motivation to live—both are classic markers of suicidal ideation. The overall tone is bleak and self‑deprecating, and the post lacks any mention of coping strategies or external support, which the model interprets as an absence of protective factors.

A confidence score of **94.4 %** is reasonable given the clear presence of these risk signals. The model is trained to treat explicit expressions of hopelessness and a desire to end life as strong indicators, so a high probability is expected. However, the post is brief and lacks contex

## 11. Launch Gradio App

In [8]:
# ── ── 10. Gradio App ─────────────────────────────────────────────────────────
print("\nLaunching Gradio app …")
print("A public share link will be displayed below.\n")

# Uncomment the line below to launch:
%run app_gradio.py

print("To launch the Gradio app, run: !python app_gradio.py")
print("\n✓ Full pipeline complete!")


Launching Gradio app …
A public share link will be displayed below.

Starting Mental Health Post Analysis System...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c27b2800f5690c87f3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


To launch the Gradio app, run: !python app_gradio.py

✓ Full pipeline complete!
